# Custom Document RAG: Context-Grounded QA Assistant 📚
*A clean, step-by-step implementation designed to build an AI assistant that answers questions accurately using exclusively your custom documents.*

The main goal of this notebook is to demonstrate a robust Retrieval-Augmented Generation (**RAG**) pipeline. You will see how raw documents are ingested, split, vectorized, and queried so that the AI generates precise, grounded answers, preventing *hallucinations* by tethering the model strictly to your provided data.

## Tech Stack & Frameworks 🛠️
- **LangChain:** For document ingestion, chunking, and orchestration.
- **ChromaDB:** An in-memory vector database requiring zero external configuration (ideal for Google Colab).
- **Hugging Face (`sentence-transformers`):** For generating high-quality local semantic embeddings without needing paid proprietary API keys.

## Start & Prerequisites 📋
1. Click the **"Open in Colab"** badge above (if hosted).
2. No paid API keys are required for the core retrieval and embedding pipeline (we use open-source local models).
3. Run the cells sequentially to test the document-grounded QA workflow.

Dependencies Installation

In [1]:
# Install required packages for Hugging Face embeddings, Chroma, and LangChain orchestration
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers transformers accelerate bitsandbytes

print("✅ All dependencies successfully installed!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateut

# Section 1 - Document Ingestion & Semantic Chunking (Code)

Here we load a sample document and split it using a smart recursive strategy, with all comments and logs in English.
In this section, we prepare our environment, securely configure our Hugging Face credentials, ingest a sample document, and split it into optimized chunks.

* **Why Hugging Face?** To run this notebook smoothly without paying for proprietary API keys, we use *open-source models* for embeddings and generation. Some models require a free Hugging Face account and token.
* **How to get your HF Token:**
  1. Create a free account on [Hugging Face](https://huggingface.co/).
  2. Go to your Settings $\rightarrow$ Access Tokens $\rightarrow$ Create new token (with `Read` permissions).
  3. In Google Colab, click on the **Key icon (Secrets)** on the left sidebar, add a secret named `HF_TOKEN`, and paste your token there.
* **Architectural Choice for Chunking:** We use `RecursiveCharacterTextSplitter`. It prioritizes natural semantic boundaries (paragraphs $\rightarrow$ sentences $\rightarrow$ words) to preserve context and prevent cutting sentences mid-way.

In [2]:
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [4]:
# Securely load Hugging Face Token from Colab Secrets
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("🔑 Hugging Face Token successfully loaded from Colab Secrets.")
except Exception as e:
    print("⚠️ Warning: 'HF_TOKEN' not found in Colab Secrets.")
    print("ℹ️ If you plan to use gated models or HF Hub features, please add your token to Colab Secrets.")
    # Optional fallback if you want to input it manually:
    # os.environ["HF_TOKEN"] = "your_hf_token_here"

🔑 Hugging Face Token successfully loaded from Colab Secrets.


In [5]:
raw_text = """
Google Colab (Colaboratory) is a hosted Jupyter notebook service that requires no setup to use and provides free access to computing resources including GPUs.
It allows developers and researchers to write and execute arbitrary Python code through the browser.

A Retrieval-Augmented Generation (RAG) architecture enhances Large Language Models (LLMs) by fetching relevant context from an external knowledge base.
The standard lifecycle involves several core stages: Ingestion, Chunking, Embedding, Vector Indexing, Retrieval, Generation, and Evaluation.

Chunking is the process of breaking large documents down into smaller, manageable segments.
Choosing the optimal chunk size and overlap is crucial: overly large chunks dilute semantic signals, while overly small chunks strip away essential surrounding context.
"""

# Wrap raw text into a LangChain Document object with metadata
source_document = Document(
    page_content=raw_text.strip(),
    metadata={"source": "rag_introduction_guide.txt", "author": "Developer"}
)

In [6]:
# Configure the RecursiveCharacterTextSplitter
chunk_size = 250
chunk_overlap = 40

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    is_separator_regex=False,
)

# Execute the chunking process
chunks = text_splitter.split_documents([source_document])

In [7]:
# 4. Clean inspection of the results
print(f"\n📄 Original document successfully split into {len(chunks)} chunks.\n" + "-"*50)
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i+1}] (Length: {len(chunk.page_content)} chars)\n{chunk.page_content}\n")


📄 Original document successfully split into 6 chunks.
--------------------------------------------------
[Chunk 1] (Length: 158 chars)
Google Colab (Colaboratory) is a hosted Jupyter notebook service that requires no setup to use and provides free access to computing resources including GPUs.

[Chunk 2] (Length: 100 chars)
It allows developers and researchers to write and execute arbitrary Python code through the browser.

[Chunk 3] (Length: 151 chars)
A Retrieval-Augmented Generation (RAG) architecture enhances Large Language Models (LLMs) by fetching relevant context from an external knowledge base.

[Chunk 4] (Length: 140 chars)
The standard lifecycle involves several core stages: Ingestion, Chunking, Embedding, Vector Indexing, Retrieval, Generation, and Evaluation.

[Chunk 5] (Length: 91 chars)
Chunking is the process of breaking large documents down into smaller, manageable segments.

[Chunk 6] (Length: 168 chars)
Choosing the optimal chunk size and overlap is crucial: overly la

# Section 2 - Embeddings & Vector Indexing (ChromaDB)

In this section, we convert our text chunks into high-dimensional vector representations that capture semantic meaning, and store them in a vector database for fast retrieval.

* **Architectural Choices:**
  * **Embeddings Model:** We use `sentence-transformers/all-MiniLM-L6-v2`. It's a lightweight, highly efficient open-source model running locally, perfect for semantic similarity tasks without requiring paid APIs.
  * **Vector Store:** We use **ChromaDB** in-memory mode. It requires zero external configuration or Docker containers, making it ideal for Google Colab notebooks.
* **Semantic Search Test:** At the end of this section, we run a test query to verify that the vector store successfully retrieves the most relevant context based on semantic proximity rather than exact keyword matching.

In [8]:
# Ensure necessary libraries are available (if not already installed)
!pip install -q langchain-huggingface langchain-chroma sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Initialize the open-source Hugging Face embedding model
# all-MiniLM-L6-v2 is fast, lightweight, and ideal for running locally on Colab CPU/GPU
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cpu"}, # Switch to "cuda" if you enabled a GPU runtime in Colab
    encode_kwargs={"normalize_embeddings": True} # Normalizes vectors for accurate cosine similarity
)

print(f" Successfully initialized embedding model: {embedding_model_name}")

# 2. Build the Vector Store using ChromaDB and index our chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_portfolio_collection"
)

print(f"✅ Vector store created! Total vectors indexed: {vector_store._collection.count()}")

# 3. Test a quick semantic similarity search
test_query = "Why is chunk size important?"
retrieved_docs = vector_store.similarity_search(test_query, k=2)

print(f"\n🔍 Test Similarity Search for query: '{test_query}'\n" + "-"*50)
for i, doc in enumerate(retrieved_docs):
    print(f"[Retrieved Result {i+1}]\n{doc.page_content}\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Successfully initialized embedding model: sentence-transformers/all-MiniLM-L6-v2
✅ Vector store created! Total vectors indexed: 6

🔍 Test Similarity Search for query: 'Why is chunk size important?'
--------------------------------------------------
[Retrieved Result 1]
Chunking is the process of breaking large documents down into smaller, manageable segments.

[Retrieved Result 2]
Choosing the optimal chunk size and overlap is crucial: overly large chunks dilute semantic signals, while overly small chunks strip away essential surrounding context.



# Section 3 - Retrieval & Search Configuration

In this section, we transition from a raw vector store to a configured **Retriever**.

* **Architectural Choice:** While a similarity search directly queries the database, wrapping the vector store into a LangChain retriever object allows us to apply advanced retrieval strategies (such as setting search parameters like `k` for the number of documents to return, or shifting to Maximum Marginal Relevance - MMR).
* **Why it matters:** The retriever acts as the bridge between the user's query and the external knowledge base, ensuring that only the most contextually relevant chunks are passed downstream to the LLM.

In [27]:
# Convert Chroma vector store into a LangChain Retriever object (top k=2 chunks)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

print("🔗 Retriever successfully configured from the vector store!")

# Test retriever with a realistic user question
user_query = "What does a RAG architecture do?"
retrieved_context = retriever.invoke(user_query)

print(f"\n🎯 User Query: '{user_query}'\n" + "-"*50)
print(f"📦 Successfully retrieved {len(retrieved_context)} context chunks:\n")

for i, doc in enumerate(retrieved_context):
    print(f"[Context Chunk {i+1}] (Source: {doc.metadata.get('source', 'Unknown')})\n{doc.page_content}\n")

🔗 Retriever successfully configured from the vector store!

🎯 User Query: 'What does a RAG architecture do?'
--------------------------------------------------
📦 Successfully retrieved 2 context chunks:

[Context Chunk 1] (Source: rag_introduction_guide.txt)
A Retrieval-Augmented Generation (RAG) architecture enhances Large Language Models (LLMs) by fetching relevant context from an external knowledge base.

[Context Chunk 2] (Source: rag_introduction_guide.txt)
The standard lifecycle involves several core stages: Ingestion, Chunking, Embedding, Vector Indexing, Retrieval, Generation, and Evaluation.



# Generation, LLM Integration & Grounded Citations 🤖

In this section, we build the core generation step of our RAG pipeline.

* **Architectural Choices:**
  * **LLM:** We use `Qwen` via Hugging Face's serverless inference or local pipelines. It offers an excellent balance between speed and reasoning capabilities for open-source models.
  * **Grounding & Prompt Engineering:** We construct a strict prompt template that forces the LLM to answer *only* using the provided context chunks. This prevents hallucinations and ensures factual accuracy.
  * **Citations:** We explicitly extract and display the source metadata alongside the generated answer to ensure total transparency.

In [37]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print(f"📥 Loading {model_id} locally with clean output configuration...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

local_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,  # 👈 Questo evita di ristampare il prompt nell'output!
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=local_pipeline)
print("🤖 Local LLM ready with clean output settings!")

📥 Loading Qwen/Qwen2.5-3B-Instruct locally with clean output configuration...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

🤖 Local LLM ready with clean output settings!


### Prompt Engineering
To prevent the LLM from hallucinating, we enforce strict **grounding**. The prompt template explicitly instructs the model to answer *only* using the provided context chunks and to admit if the information is missing.

In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define prompt structure optimized for instruction following
template = """You are an expert AI assistant. Use the following pieces of retrieved context to answer the question at the end.
If you do not know the answer based on the context, state clearly that you cannot find the information in the provided documents. Do not make up facts.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. Construct the RAG Chain using LangChain Expression Language (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("⛓️ RAG Chain successfully assembled via LCEL!")

⛓️ RAG Chain successfully assembled via LCEL!


### Query execution
We combine our retriever, formatter, prompt, and LLM into a unified pipeline using **LangChain Expression Language (LCEL)**. Finally, we execute a query and output both the generated answer and the source citations.

In [39]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define prompt using Qwen 2.5's native chat/instruction tokens (<|im_start|> / <|im_end|>)
template = """<|im_start|>system
You are an expert AI assistant. Use the following pieces of retrieved context to answer the question at the end. If you do not know the answer based on the context, state clearly that you cannot find the information in the provided documents. Do not make up facts.<|im_end|>
<|im_start|>user
Context:
{context}

Question: {question}<|im_end|>
<|im_start|>assistant
"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. Construct the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 3. Test execution
final_query = "What is chunking and why is the choice of chunk size important?"

print(f"\n💬 Running Clean RAG Chain for Question: '{final_query}'\n" + "-"*60)
response = rag_chain.invoke(final_query)

print(f"\n✨ Generated Answer:\n{response.strip()}\n" + "-"*60)

# Display sources
source_docs = retriever.invoke(final_query)
print("\n📚 Sources & Citations used for this answer:")
for i, doc in enumerate(source_docs):
    print(f"- Source [{i+1}]: {doc.metadata.get('source', 'Unknown Document')} (Length: {len(doc.page_content)} chars)")


💬 Running Clean RAG Chain for Question: 'What is chunking and why is the choice of chunk size important?'
------------------------------------------------------------

✨ Generated Answer:
Chunking is the process of dividing large documents into smaller, more manageable parts. The selection of an appropriate chunk size is critical because if the chunks are too large, they can lose the meaning or context within the document (overly large chunks dilute semantic signals). Conversely, if the chunks are too small, they may lack the necessary surrounding context (overly small chunks strip away essential surrounding context). Therefore, choosing the right chunk size helps maintain both the integrity of the document's content and its contextual understanding.
------------------------------------------------------------

📚 Sources & Citations used for this answer:
- Source [1]: rag_introduction_guide.txt (Length: 91 chars)
- Source [2]: rag_introduction_guide.txt (Length: 168 chars)


# RAG Evaluation (LLM-as-a-Judge)

When testing a RAG system with **custom user documents**, traditional metrics (like BLEU or ROUGE) fail because they require a pre-built ground-truth dataset.

To solve this, we implement a **reference-free evaluation function**. Whenever a user asks a question on their own custom text, this function dynamically audits the output using two core metrics:
1. **Faithfulness (Grounding):** Checks if the answer is strictly derived from the retrieved context (preventing hallucinations).
2. **Answer Relevance:** Checks if the response actually and directly answers the user's question.

In [38]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define evaluation prompt template using Qwen native tags
eval_template = """<|im_start|>system
You are an objective AI evaluation judge. Evaluate the quality of the RAG system's response based on the provided context and question. Do not assume external facts.<|im_end|>
<|im_start|>user
Criteria (score from 1 to 5 for each):
1. Faithfulness: Is the generated answer strictly supported by the context? (1 = hallucinations, 5 = perfectly grounded).
2. Answer Relevance: Does the response directly answer the user's question? (1 = off-topic, 5 = directly answers).

Context:
{context}

Question: {question}

Generated Response:
{response}

Provide your evaluation in this exact format:
- Faithfulness Score: [1-5]/5 - [Short justification]
- Answer Relevance Score: [1-5]/5 - [Short justification]<|im_end|>
<|im_start|>assistant
"""

eval_prompt = PromptTemplate.from_template(eval_template)
eval_chain = eval_prompt | llm | StrOutputParser()

# 2. Evaluation execution function
def evaluate_rag_output(query, retrieved_chunks, generated_response):
    formatted_context = "\n\n".join([doc.page_content for doc in retrieved_chunks])

    print(f"📊 Evaluating RAG output for query: '{query}'...\n" + "-"*50)

    evaluation_result = eval_chain.invoke({
        "context": formatted_context,
        "question": query,
        "response": generated_response
    })

    return evaluation_result

# 3. Run evaluation
eval_output = evaluate_rag_output(final_query, source_docs, response)

print(eval_output.strip())
print("-" * 50)
print("✅ Evaluation completed successfully!")

📊 Evaluating RAG output for query: 'What is chunking and why is the choice of chunk size important?'...
--------------------------------------------------
- Faithfulness Score: 5/5 - The response is fully grounded in the provided context.
- Answer Relevance Score: 5/5 - The response directly addresses the question about what chunking is and why the choice of chunk size is important.
--------------------------------------------------
✅ Evaluation completed successfully!
